# Best-Fitting Clothing Project 


## Imports


In [40]:
import os
import json
import re
import time

import numpy as np
import pandas as pd
import scipy.sparse as sp

import spacy
from tqdm import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

from xgboost import XGBClassifier
import joblib

import torch
from torchvision.models import resnet101, ResNet101_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image

---
# Part 1 — Data Preparation
Loads RTR, ModCloth, and the Fashion Product Images styles metadata, cleans
each one (missing values, unit conversions, outliers, duplicates), and
merges RTR + ModCloth into `merged_df`.


In [2]:
mo_df = pd.read_json(r"..\data\modcloth_final_data.json", lines=True)
re_df = pd.read_json(r"..\data\renttherunway_final_data.json", lines=True)
st_df = pd.read_csv(r"..\data\styles.csv", on_bad_lines="skip")

In [3]:
st_df["baseColour"] = st_df["baseColour"].fillna(st_df["baseColour"].mode()[0])
st_df["season"] = st_df["season"].fillna(st_df["season"].mode()[0])
st_df["year"] = st_df["year"].fillna(st_df["year"].median())
st_df["usage"] = st_df["usage"].fillna(st_df["usage"].mode()[0])
st_df["productDisplayName"] = st_df["productDisplayName"].fillna("Unknown")

In [4]:
re_df["age"] = re_df["age"].fillna(re_df["age"].median())
re_df["height"] = re_df["height"].fillna(re_df["height"].mode()[0])
re_df["body type"] = re_df["body type"].fillna(re_df["body type"].mode()[0])
re_df["review_summary"] = re_df["review_summary"].fillna("No Summary")
re_df["weight"] = re_df["weight"].fillna(re_df["weight"].mode()[0])
re_df["bust size"] = re_df["bust size"].fillna(re_df["bust size"].mode()[0])
re_df["rating"] = re_df["rating"].fillna(re_df["rating"].median())
re_df["rented for"] = re_df["rented for"].fillna(re_df["rented for"].mode()[0])
re_df["review_text"] = re_df["review_text"].fillna("No Review")

In [5]:
def convert_height(x):
    if pd.isna(x):
        return None
    feet = int(x.split("'")[0])
    inches = int(x.split("'")[1].replace('"', '').strip())
    cm = (feet * 30.48) + (inches * 2.54)
    return round(cm)

re_df["height_cm"] = re_df["height"].apply(convert_height)
re_df.drop("height", axis=1, inplace=True)

In [6]:
re_df["weight_lbs"] = re_df["weight"].str.replace("lbs", "").astype(int)
re_df["weight_kg"] = (
    re_df["weight"]
    .str.replace("lbs", "")
    .astype(int) * 0.453592
)
re_df["weight_kg"] = re_df["weight_kg"].round().astype(int)
re_df.drop("weight", axis=1, inplace=True)
re_df.drop("weight_lbs", axis=1, inplace=True)

In [7]:
re_df = re_df.drop_duplicates()
re_df.loc[re_df["height_cm"] > 200, "height_cm"] = None
re_df["height_cm"] = re_df["height_cm"].fillna(re_df["height_cm"].median())
re_df.loc[re_df["age"] > 80, "age"] = None
re_df["age"] = re_df["age"].fillna(re_df["age"].median())

In [8]:
def height_to_cm(x):
    if pd.isna(x):
        return None
    x = x.replace("ft", "").replace("in", "")
    parts = x.split()
    feet = int(parts[0])
    inches = int(parts[1]) if len(parts) > 1 else 0
    return round((feet * 30.48) + (inches * 2.54))

mo_df["height_cm"] = mo_df["height"].apply(height_to_cm)
mo_df.drop("height", axis=1, inplace=True)

In [9]:
mo_df["review_summary"] = mo_df["review_summary"].fillna("No Summary")
mo_df["review_text"] = mo_df["review_text"].fillna("No Review")
mo_df["height_cm"] = mo_df["height_cm"].fillna(mo_df["height_cm"].median())
mo_df["cup size"] = mo_df["cup size"].fillna(mo_df["cup size"].mode()[0])
mo_df["length"] = mo_df["length"].fillna(mo_df["length"].mode()[0])
mo_df["quality"] = mo_df["quality"].fillna(mo_df["quality"].mode()[0])

In [10]:
mo_df = mo_df.drop_duplicates()
mo_df = mo_df[(mo_df["height_cm"] >= 140) & (mo_df["height_cm"] <= 210)]
mo_df.loc[mo_df["size"] == 0, "size"] = mo_df["size"].median()

median_height = mo_df["height_cm"].median()
mo_df.loc[
    (mo_df["height_cm"] < 140) |
    (mo_df["height_cm"] > 210),
    "height_cm"
] = median_height

mo_df.loc[
    (mo_df["height_cm"] < 140) |
    (mo_df["height_cm"] > 210),
    "height_cm"
] = mo_df["height_cm"].median()

In [11]:
re_df["source"] = "renttherunway"
mo_df["source"] = "modcloth"

print("re_df columns:", list(re_df.columns))
print("mo_df columns:", list(mo_df.columns))

merged_df = pd.concat([re_df, mo_df], ignore_index=True, sort=False)

print("re_df shape:", re_df.shape)
print("mo_df shape:", mo_df.shape)
print("merged_df shape:", merged_df.shape)

re_df columns: ['fit', 'user_id', 'bust size', 'item_id', 'rating', 'rented for', 'review_text', 'body type', 'review_summary', 'category', 'size', 'age', 'review_date', 'height_cm', 'weight_kg', 'source']
mo_df columns: ['item_id', 'waist', 'size', 'quality', 'cup size', 'hips', 'bra size', 'category', 'bust', 'user_name', 'length', 'fit', 'user_id', 'shoe size', 'shoe width', 'review_summary', 'review_text', 'height_cm', 'source']
re_df shape: (192355, 16)
mo_df shape: (82349, 19)
merged_df shape: (274704, 26)


---
#  Visual Feature Extraction & Fusion
Links product images to their metadata, runs them through a frozen
ResNet101 backbone to extract 2048-dim visual features per image, averages
those per `articleType`, then maps RTR/ModCloth's category names onto the
image categories.


In [12]:
image_dir = r"..\data\images"
jpg_files = [
    f for f in os.listdir(image_dir)
    if f.lower().endswith(".jpg")
]

image_df = pd.DataFrame({"filename": jpg_files})

image_df["id"] = pd.to_numeric(
    image_df["filename"].str.replace(".jpg", "", regex=False),
    errors="coerce"
)

image_df = image_df.merge(
    st_df,
    on="id",
    how="inner"
)

print("Images linked to styles:", len(image_df))

Images linked to styles: 44419


In [13]:
TORCH_CACHE_DIR = r"D:\torch_cache"
os.makedirs(TORCH_CACHE_DIR, exist_ok=True)

os.environ["TORCH_HOME"] = TORCH_CACHE_DIR
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = ResNet101_Weights.DEFAULT

model = resnet101(weights=weights)
model = model.to(device)
model.eval()

transform = weights.transforms()

print("Using device:", device)

Downloading: "https://download.pytorch.org/models/resnet101-cd907fc2.pth" to D:\torch_cache\hub\checkpoints\resnet101-cd907fc2.pth


100%|██████████| 171M/171M [00:50<00:00, 3.58MB/s] 


Using device: cuda


In [14]:
feature_extractor = torch.nn.Sequential(
    *list(model.children())[:-1]
)

feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, ker

In [15]:
class ClothingImageDataset(Dataset):

    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = os.path.join(
            self.image_dir,
            row["filename"]
        )

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, row["id"], row["articleType"]

In [16]:
dataset = ClothingImageDataset(
    image_df,
    image_dir,
    transform=transform
)

dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Number of images:", len(dataset))

Number of images: 44419


In [17]:
all_features = []
all_ids = []
all_categories = []

with torch.no_grad():

    for images, ids, categories in dataloader:

        images = images.to(device)

        features = feature_extractor(images)

        features = features.flatten(1)

        all_features.append(features.cpu())
        all_ids.extend(ids.numpy())
        all_categories.extend(categories)

print("Feature extraction complete.")

Feature extraction complete.


In [18]:
features_tensor = torch.cat(all_features, dim=0)

features_df = pd.DataFrame(
    features_tensor.numpy(),
    columns=[f"image_feature_{i+1}" for i in range(2048)]
)

features_df["id"] = all_ids
features_df["articleType"] = all_categories

print("Feature shape:", features_df.shape)

Feature shape: (44419, 2050)


In [19]:
image_feature_columns = [
    f"image_feature_{i+1}" for i in range(2048)
]

category_features = features_df.groupby(
    "articleType"
)[image_feature_columns].mean()

print("Category feature shape:", category_features.shape)

Category feature shape: (142, 2048)


In [20]:
category_mapping = {
    "dress": "Dresses", "dresses": "Dresses", "gown": "Dresses", "ballgown": "Dresses",
    "sheath": "Dresses", "shift": "Dresses", "shirtdress": "Dresses", "frock": "Dresses",
    "maxi": "Dresses", "midi": "Dresses", "mini": "Dresses", "wedding": "Dresses",

    "top": "Tops", "tops": "Tops", "blouse": "Tops", "tank": "Tops", "turtleneck": "Tops",

    "tee": "Tshirts", "t-shirt": "Tshirts", "henley": "Tshirts", "crewneck": "Tshirts",

    "shirt": "Shirts", "buttondown": "Shirts",

    "sweater": "Sweaters", "cardigan": "Sweaters", "knit": "Sweaters", "pullover": "Sweaters",

    "sweatshirt": "Sweatshirts", "sweatershirt": "Sweatshirts", "hoodie": "Sweatshirts",

    "pants": "Trousers", "pant": "Trousers", "trouser": "Trousers", "trousers": "Trousers",
    "culotte": "Trousers", "culottes": "Trousers", "bottoms": "Trousers",

    "jeans": "Jeans",

    "legging": "Leggings", "leggings": "Leggings",

    "jacket": "Jackets", "outerwear": "Jackets", "bomber": "Jackets", "parka": "Jackets",
    "trench": "Jackets", "duster": "Jackets", "blouson": "Jackets", "coat": "Jackets",
    "peacoat": "Jackets", "overcoat": "Jackets",

    "blazer": "Blazers",

    "skirt": "Skirts", "skirts": "Skirts", "skort": "Skirts",

    "jumpsuit": "Jumpsuit", "romper": "Rompers", "combo": "Clothing Set", "overalls": "Clothing Set",

    "tunic": "Tunics", "kimono": "Tunics", "poncho": "Tunics", "cape": "Tunics",

    "kaftan": "Kurtas", "caftan": "Kurtas",

    "vest": "Waistcoat",

    "suit": "Clothing Set", "cami": "Camisoles", "tight": "Tights",
    "jogger": "Track Pants", "sweatpants": "Track Pants",
}

In [21]:
merged_df["image_category"] = merged_df["category"].map(category_mapping)

print("Total reviews:", len(merged_df))
print("Mapped:", merged_df["image_category"].notna().sum())
print("Unmapped:", merged_df["image_category"].isna().sum())

category_features.to_csv("../data/category_image_features.csv")

merged_df.to_parquet("../data/merged_with_image_category.parquet", index=False)

print("Phase 2 files saved successfully.")
print("category_features shape:", category_features.shape)
print("merged_df shape:", merged_df.shape)

Total reviews: 274704
Mapped: 250229
Unmapped: 24475
Phase 2 files saved successfully.
category_features shape: (142, 2048)
merged_df shape: (274704, 27)


---
#  Text Feature Extraction
Cleans review text (regex → lowercase → tokenize → lemmatize → remove
stopwords) with spaCy, then converts it to 300 TF-IDF features.


In [22]:
final_df = pd.read_parquet("../data/merged_with_image_category.parquet")
print(final_df.shape)

(274704, 27)


In [23]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def clean_text(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"[^a-zA-Z\s]", "", text.lower())

pre_cleaned = [clean_text(t) for t in final_df["review_text"]]

cleaned = []
t0 = time.time()
for doc in tqdm(nlp.pipe(pre_cleaned, batch_size=500), total=len(pre_cleaned), desc="Cleaning reviews"):
    tokens = [token.lemma_ for token in doc if not token.is_stop and token.lemma_.strip()]
    cleaned.append(" ".join(tokens))

final_df["clean_review"] = cleaned
print(f"Done | {time.time()-t0:.1f}s")

Cleaning reviews: 100%|██████████| 274704/274704 [12:59<00:00, 352.37it/s] 

Done | 779.6s


In [24]:
MAX_TFIDF_FEATURES = 300

vectorizer = TfidfVectorizer(max_features=MAX_TFIDF_FEATURES)
tfidf_matrix = vectorizer.fit_transform(final_df["clean_review"])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix shape: (274704, 300)


In [25]:
sp.save_npz("../data/text_features.npz", tfidf_matrix)
joblib.dump(vectorizer, "../data/tfidf_vectorizer.pkl")

final_df.to_parquet("../data/final_df_with_clean_text.parquet")

print("Saved: text_features.npz, tfidf_vectorizer.pkl, final_df_with_clean_text.parquet")

Saved: text_features.npz, tfidf_vectorizer.pkl, final_df_with_clean_text.parquet


---
#  Feature Fusion
Combines visual (2048), text (300), numeric (3: height/weight/age), and
categorical (30, padded/truncated) features into one final matrix, and
saves the `fit` target labels.


In [26]:
print("Stage 4 started")

final_df = pd.read_parquet("../data/final_df_with_clean_text.parquet")
tfidf_matrix = sp.load_npz("../data/text_features.npz").tocsr()
category_features = pd.read_csv("../data/category_image_features.csv", index_col=0)

print("DataFrame shape:", final_df.shape)
print("Text features shape:", tfidf_matrix.shape)
print("Image category features shape:", category_features.shape)

if category_features.shape[1] != 2048:
    raise ValueError(f"Expected 2048 image features, found {category_features.shape[1]}")

if tfidf_matrix.shape[1] != 300:
    raise ValueError(f"Expected 300 TF-IDF features, found {tfidf_matrix.shape[1]}")

print("✓ Image features = 2048")
print("✓ Text features = 300")

if "image_category" not in final_df.columns:
    raise ValueError("image_category is missing from final_df.")

image_block = category_features.reindex(
    final_df["image_category"]
).to_numpy(dtype=np.float32)

missing_mask = final_df["image_category"].isna()

image_block[missing_mask] = 0.0

print("Image block:", image_block.shape)
print("Rows without image category:", missing_mask.sum())

Stage 4 started
DataFrame shape: (274704, 28)
Text features shape: (274704, 300)
Image category features shape: (142, 2048)
✓ Image features = 2048
✓ Text features = 300
Image block: (274704, 2048)
Rows without image category: 24475


In [27]:
def parse_height_cm(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    m = re.search(r"(\d+)\s*(?:ft|')\s*(\d+)?\s*(?:in|\")?", s)
    if m:
        return float(m.group(1)) * 30.48 + float(m.group(2) or 0) * 2.54
    m = re.search(r"(\d+(?:\.\d+)?)\s*cm", s)
    if m:
        return float(m.group(1))
    try:
        x = float(s)
        return x * 30.48 if x < 10 else x
    except:
        return np.nan

def parse_weight_kg(value):
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    m = re.search(r"(\d+(?:\.\d+)?)\s*(?:lbs?|pounds?)", s)
    if m:
        return float(m.group(1)) * 0.45359237
    m = re.search(r"(\d+(?:\.\d+)?)\s*kg", s)
    if m:
        return float(m.group(1))
    try:
        x = float(s)
        return x * 0.45359237 if x > 100 else x
    except:
        return np.nan

def first_existing_column(df, names):
    for name in names:
        if name in df.columns:
            return name
    return None

height_col = first_existing_column(final_df, ["height_cm", "height"])
weight_col = first_existing_column(final_df, ["weight_kg", "weight"])
age_col = first_existing_column(final_df, ["age"])

if height_col is None or weight_col is None or age_col is None:
    raise ValueError(f"Stage 4 requires height, weight and age. Found: {list(final_df.columns)}")

numeric_df = pd.DataFrame({
    "height": final_df[height_col].apply(parse_height_cm),
    "weight": final_df[weight_col].apply(parse_weight_kg),
    "age": pd.to_numeric(final_df[age_col], errors="coerce")
})

numeric_df = numeric_df.fillna(numeric_df.median())

scaler = StandardScaler()
numeric_block = scaler.fit_transform(numeric_df).astype(np.float32)
joblib.dump(scaler, "../data/numeric_scaler.pkl")

print("Numeric block:", numeric_block.shape)

Numeric block: (274704, 3)


In [28]:
categorical_candidates = [
    "bust size", "bra size", "cup size", "body type", "rented for",
    "category", "length", "size", "quality", "brand", "gender",
    "masterCategory", "subCategory", "articleType", "baseColour",
    "season", "usage"
]

available_categorical = [c for c in categorical_candidates if c in final_df.columns]
if not available_categorical:
    raise ValueError("No categorical columns were found.")

categorical_df = final_df[available_categorical].copy()
for col in categorical_df.columns:
    categorical_df[col] = categorical_df[col].fillna("Unknown").astype(str).str.strip()

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32)
categorical_encoded = encoder.fit_transform(categorical_df)
joblib.dump(encoder, "../data/categorical_encoder.pkl")

print("Categorical columns:", available_categorical)
print("Natural one-hot shape:", categorical_encoded.shape)

TARGET_CATEGORICAL_FEATURES = 30

if categorical_encoded.shape[1] >= TARGET_CATEGORICAL_FEATURES:
    categorical_block = categorical_encoded[:, :TARGET_CATEGORICAL_FEATURES]
else:
    padding = sp.csr_matrix(
        (categorical_encoded.shape[0], TARGET_CATEGORICAL_FEATURES - categorical_encoded.shape[1]),
        dtype=np.float32
    )
    categorical_block = sp.hstack([categorical_encoded, padding], format="csr")

print("Categorical block:", categorical_block.shape)

Categorical columns: ['bust size', 'bra size', 'cup size', 'body type', 'rented for', 'category', 'length', 'size', 'quality']
Natural one-hot shape: (274704, 294)
Categorical block: (274704, 30)


In [29]:
n_samples = len(final_df)

for name, block in {
    "image": image_block,
    "text": tfidf_matrix,
    "numeric": numeric_block,
    "categorical": categorical_block
}.items():
    if block.shape[0] != n_samples:
        raise ValueError(f"{name} block row count does not match final_df")

print("✓ All blocks have", n_samples, "rows")

image_sparse = sp.csr_matrix(image_block, dtype=np.float32)
numeric_sparse = sp.csr_matrix(numeric_block, dtype=np.float32)

final_fused_features = sp.hstack(
    [image_sparse, tfidf_matrix, numeric_sparse, categorical_block],
    format="csr",
    dtype=np.float32
)

expected_features = 2048 + 300 + 3 + 30
print("Final fused matrix shape:", final_fused_features.shape)
print("Expected features:", expected_features)

if final_fused_features.shape[1] != expected_features:
    raise ValueError(f"Expected {expected_features} features, found {final_fused_features.shape[1]}")

sp.save_npz("../data/final_fused_features.npz", final_fused_features)
print("Saved: ../data/final_fused_features.npz")

✓ All blocks have 274704 rows
Final fused matrix shape: (274704, 2381)
Expected features: 2381
Saved: ../data/final_fused_features.npz


In [30]:
if "fit" in final_df.columns:
    fit_labels = final_df["fit"].astype(str).str.strip().str.lower()
elif "fit_label" in final_df.columns:
    fit_labels = final_df["fit_label"].astype(str).str.strip().str.lower()
else:
    raise ValueError("No 'fit' or 'fit_label' column found.")

pd.DataFrame({"fit_label": fit_labels}).to_csv("../data/fit_labels.csv", index=False)
print("Saved: ../data/fit_labels.csv")
print(fit_labels.value_counts())

Saved: ../data/fit_labels.csv
fit
fit      198381
small     38632
large     37691
Name: count, dtype: int64


---
#  Model Training
Loads the fused features and labels, encodes the target, splits
train/test, then runs `GridSearchCV` over an `XGBClassifier` to find and
train the best model.


In [31]:
X = sp.load_npz("../data/final_fused_features.npz")

print("Feature matrix loaded successfully.")
print("X shape:", X.shape)

labels_df = pd.read_csv("../data/fit_labels.csv")

y = labels_df["fit_label"].astype(str).str.strip().str.lower()

print("\nLabels loaded successfully.")
print("Number of labels:", len(y))

print("\nUnique fit labels:")
print(y.value_counts())

Feature matrix loaded successfully.
X shape: (274704, 2381)

Labels loaded successfully.
Number of labels: 274704

Unique fit labels:
fit_label
fit      198381
small     38632
large     37691
Name: count, dtype: int64


In [32]:
if X.shape[0] != len(y):
    raise ValueError(
        f"Mismatch between features and labels.\n"
        f"Features: {X.shape[0]}\n"
        f"Labels: {len(y)}"
    )

print("Features and labels have matching row counts.")

Features and labels have matching row counts.


In [33]:
label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Label Encoding:")

for index, label in enumerate(label_encoder.classes_):
    print(f"{label} --> {index}")

joblib.dump(
    label_encoder,
    "../data/fit_label_encoder.pkl"
)

print("\nLabel encoder saved successfully.")

Label Encoding:
fit --> 0
large --> 1
small --> 2

Label encoder saved successfully.


In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("Training set shape:", X_train.shape)
print("Testing set shape :", X_test.shape)

print("\nTraining samples:", len(y_train))
print("Testing samples :", len(y_test))

Training set shape: (219763, 2381)
Testing set shape : (54941, 2381)

Training samples: 219763
Testing samples : 54941


In [45]:
xgb_model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=len(label_encoder.classes_),
    tree_method="hist",
    device="cuda",
    random_state=42
)

print("Base XGBoost model created.")

Base XGBoost model created.


In [46]:
param_grid = {
    "learning_rate": [0.05, 0.1],
    "max_depth": [3, 5, 7],
    "n_estimators": [100, 200],
    "subsample": [0.8, 1.0]
}

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=12,
    scoring="f1_weighted",
    cv=2,
    verbose=2,
    n_jobs=1,
    random_state=42
)

print("Starting Randomized Search...")

random_search.fit(
    X_train,
    y_train
)

Starting Randomized Search...
Fitting 2 folds for each of 12 candidates, totalling 24 fits


C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\xgboost\core.py:729: UserWarning: [08:11:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


[CV] END learning_rate=0.05, max_depth=7, n_estimators=100, subsample=0.8; total time=  35.7s
[CV] END learning_rate=0.05, max_depth=7, n_estimators=100, subsample=0.8; total time=  33.2s
[CV] END learning_rate=0.1, max_depth=5, n_estimators=100, subsample=0.8; total time=  21.6s
[CV] END learning_rate=0.1, max_depth=5, n_estimators=100, subsample=0.8; total time=  21.3s
[CV] END learning_rate=0.05, max_depth=3, n_estimators=100, subsample=0.8; total time=  18.0s
[CV] END learning_rate=0.05, max_depth=3, n_estimators=100, subsample=0.8; total time=  18.2s
[CV] END learning_rate=0.1, max_depth=5, n_estimators=200, subsample=0.8; total time=  30.4s
[CV] END learning_rate=0.1, max_depth=5, n_estimators=200, subsample=0.8; total time=  30.2s
[CV] END learning_rate=0.05, max_depth=7, n_estimators=200, subsample=1.0; total time=  44.3s
[CV] END learning_rate=0.05, max_depth=7, n_estimators=200, subsample=1.0; total time=  43.8s
[CV] END learning_rate=0.05, max_depth=7, n_estimators=100, subs

,estimator,"XGBClassifier..._class=3, ...)"
,param_distributions,"{'learning_rate': [0.05, 0.1], 'max_depth': [3, 5, ...], 'n_estimators': [100, 200], 'subsample': [0.8, 1.0]}"
,n_iter,12
,scoring,'f1_weighted'
,n_jobs,1
,refit,True
,cv,2
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [47]:
print("Best parameters:", random_search.best_params_)
print("Best F1:", random_search.best_score_)

Best parameters: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}
Best F1: 0.7569017440237973


In [49]:
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest Cross Validation Score:")
print(random_search.best_score_)

Best Parameters:
{'subsample': 0.8, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}

Best Cross Validation Score:
0.7569017440237973


In [51]:
best_xgb_model = random_search.best_estimator_
print("Best XGBoost model selected.")

Best XGBoost model selected.


In [52]:
print("Training the final XGBoost model...")

best_xgb_model.fit(
    X_train,
    y_train
)

print("Final model training completed.")

Training the final XGBoost model...
Final model training completed.


In [53]:
joblib.dump(
    best_xgb_model,
    "../data/best_xgboost_fit_model.pkl"
)

print("XGBoost model saved successfully.")

XGBoost model saved successfully.


---
#  Deployment
** `load_inference_artifacts()` separately inside
`predict_fit_for_all_garments()` *and* inside `predict_fit_from_photo_only()`,
meaning every prediction reloaded the model, scaler, encoder, vectorizer,
and category features from disk again. This version loads everything
**once** in `run_model_one_pipeline()` and passes it through.

**What this does:**
1. Loads a photo.
2. Estimates body type from pose landmarks (MediaPipe).
3. Detects individual garments in the photo (YOLOS-Fashionpedia), filtered
   to real clothing categories only.
4. For each detected garment (or a fallback category if none are detected),
   builds the same 2048+300+3+30 feature vector used in training and
   predicts fit with the trained XGBoost model.


In [66]:
import urllib.request
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
from transformers import YolosImageProcessor, YolosForObjectDetection

DATA_DIR = r"..\data"

MODEL_PATH = os.path.join(DATA_DIR, "best_xgboost_fit_model.pkl")
SCALER_PATH = os.path.join(DATA_DIR, "numeric_scaler.pkl")
ENCODER_PATH = os.path.join(DATA_DIR, "categorical_encoder.pkl")
VECTORIZER_PATH = os.path.join(DATA_DIR, "tfidf_vectorizer.pkl")
LABEL_ENCODER_PATH = os.path.join(DATA_DIR, "fit_label_encoder.pkl")
CATEGORY_FEATURES_PATH = os.path.join(DATA_DIR, "category_image_features.csv")
FALLBACKS_PATH = os.path.join(DATA_DIR, "training_fallbacks.json")
FINAL_DF_PATH = os.path.join(DATA_DIR, "final_df_with_clean_text.parquet")

TARGET_CATEGORICAL_FEATURES = 30

POSE_MODEL_PATH = os.path.join(DATA_DIR, "pose_landmarker.task")
POSE_MODEL_URL = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task"

GARMENT_LABELS = {
    "shirt, blouse", "top, t-shirt, sweatshirt", "sweater", "cardigan", "jacket",
    "vest", "pants", "shorts", "skirt", "coat", "dress", "jumpsuit", "cape",
}

FASHIONPEDIA_TO_INFO = {
    "shirt, blouse": ("shirt", "Shirts"),
    "top, t-shirt, sweatshirt": ("top", "Tops"),
    "sweater": ("sweater", "Sweaters"),
    "cardigan": ("cardigan", "Sweaters"),
    "jacket": ("jacket", "Jackets"),
    "vest": ("vest", "Waistcoat"),
    "pants": ("jeans", "Jeans"),
    "shorts": ("shorts", "Shorts"),
    "skirt": ("skirt", "Skirts"),
    "coat": ("jacket", "Jackets"),
    "dress": ("dress", "Dresses"),
    "jumpsuit": ("jumpsuit", "Jumpsuit"),
    "cape": ("tunic", "Tunics"),
}

In [57]:
def check_required_files():
    required = {
        "XGBoost model": MODEL_PATH,
        "Numeric scaler": SCALER_PATH,
        "Categorical encoder": ENCODER_PATH,
        "TF-IDF vectorizer": VECTORIZER_PATH,
        "Fit label encoder": LABEL_ENCODER_PATH,
        "Category image features CSV": CATEGORY_FEATURES_PATH,
    }

    missing = []
    for name, path in required.items():
        if not os.path.exists(path):
            missing.append(f"  - {name}: {path}")

    if not os.path.exists(FALLBACKS_PATH) and not os.path.exists(FINAL_DF_PATH):
        missing.append(
            f"  - Either training_fallbacks.json OR final_df_with_clean_text.parquet must exist: "
            f"{FALLBACKS_PATH} / {FINAL_DF_PATH}"
        )

    if missing:
        raise FileNotFoundError(
            "Missing required files in DATA_DIR ({}):\n{}\n\n"
            "Copy these files from your Parts 1-5 output folder into DATA_DIR, "
            "or change DATA_DIR to point to where they actually are."
            .format(DATA_DIR, "\n".join(missing))
        )

    print("All required artifact files found in:", DATA_DIR)


def ensure_pose_model():
    if not os.path.exists(POSE_MODEL_PATH):
        urllib.request.urlretrieve(POSE_MODEL_URL, POSE_MODEL_PATH)
    return POSE_MODEL_PATH


def build_pose_landmarker():
    base_options = mp_python.BaseOptions(model_asset_path=ensure_pose_model())
    options = vision.PoseLandmarkerOptions(base_options=base_options, output_segmentation_masks=False)
    return vision.PoseLandmarker.create_from_options(options)


def build_yolos():
    processor = YolosImageProcessor.from_pretrained("valentinafeve/yolos-fashionpedia")
    detection_model = YolosForObjectDetection.from_pretrained("valentinafeve/yolos-fashionpedia")
    return processor, detection_model


def build_spacy_nlp():
    return spacy.load("en_core_web_sm", disable=["parser", "ner"])

In [58]:
def generate_training_fallbacks(categorical_columns):
    df = pd.read_parquet(FINAL_DF_PATH)

    numeric_defaults = {
        "height_cm_median": float(df["height_cm"].median()),
        "weight_kg_median": float(df["weight_kg"].median()),
        "age_median": float(df["age"].median()),
    }

    categorical_defaults = {}
    for col in categorical_columns:
        if col in df.columns:
            series = df[col].fillna("Unknown").astype(str).str.strip()
            categorical_defaults[col] = series.mode().iloc[0]
        else:
            categorical_defaults[col] = "Unknown"

    fallbacks = {"numeric": numeric_defaults, "categorical": categorical_defaults}

    with open(FALLBACKS_PATH, "w") as f:
        json.dump(fallbacks, f, indent=2)

    return fallbacks


def load_inference_artifacts():
    """Loads every model/preprocessing artifact ONCE. Called a single time
    by run_model_one_pipeline() and passed down to every prediction call,
    instead of being reloaded from disk on every single prediction."""
    scaler = joblib.load(SCALER_PATH)
    encoder = joblib.load(ENCODER_PATH)
    vectorizer = joblib.load(VECTORIZER_PATH)
    label_encoder = joblib.load(LABEL_ENCODER_PATH)
    category_features = pd.read_csv(CATEGORY_FEATURES_PATH, index_col=0)
    model = joblib.load(MODEL_PATH)

    categorical_columns = list(encoder.feature_names_in_)

    if os.path.exists(FALLBACKS_PATH):
        with open(FALLBACKS_PATH) as f:
            fallbacks = json.load(f)
    else:
        fallbacks = generate_training_fallbacks(categorical_columns)

    return {
        "scaler": scaler,
        "encoder": encoder,
        "vectorizer": vectorizer,
        "label_encoder": label_encoder,
        "category_features": category_features,
        "model": model,
        "fallbacks": fallbacks,
        "categorical_columns": categorical_columns,
    }

In [ ]:
def get_category_vector(category, category_features):
    if category not in category_features.index:
        raise ValueError(
            f"No visual vector for category '{category}'. "
            f"Known: {sorted(category_features.index.tolist())}"
        )
    return category_features.loc[category].to_numpy(dtype=np.float32).reshape(1, -1)


def clean_review_text(text, nlp):
    pre = re.sub(r"[^a-zA-Z\s]", "", str(text).lower())
    doc = nlp(pre)
    tokens = [t.lemma_ for t in doc if not t.is_stop and t.lemma_.strip()]
    return " ".join(tokens)


def estimate_body_type_from_photo(image_path):
    mp_image = mp.Image.create_from_file(image_path)
    result = pose_landmarker.detect(mp_image)

    if not result.pose_landmarks:
        print("No person detected — defaulting body_type to 'unknown'")
        return "unknown"

   lm = result.pose_landmarks[0]
    shoulder_width = abs(lm[11].x - lm[12].x)
    hip_width = abs(lm[23].x - lm[24].x)
    ratio = shoulder_width / hip_width if hip_width > 0 else 1.0

    if ratio > 1.8:
        body_type = "athletic"
    elif ratio <= 1.8:
        body_type = "pear"
    else:
        body_type = "hourglass"

    print(f"Estimated body_type: {body_type} (shoulder/hip ratio: {ratio:.2f}, thresholds unvalidated)")
    return body_type

In [60]:
def detect_and_classify_all_garments(image_path, processor, detection_model, category_features, confidence_threshold=0.5):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")

    with torch.no_grad():
        outputs = detection_model(**inputs)

    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(
        outputs, threshold=confidence_threshold, target_sizes=target_sizes
    )[0]

    detections = []
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        fashionpedia_label = detection_model.config.id2label[label.item()]

        if fashionpedia_label not in GARMENT_LABELS:
            continue

        info = FASHIONPEDIA_TO_INFO.get(fashionpedia_label)
        if info is None:
            continue

        raw_category, image_category = info
        if image_category not in category_features.index:
            continue

        box = [round(i) for i in box.tolist()]
        detections.append({
            "fashionpedia_label": fashionpedia_label,
            "confidence": round(score.item(), 3),
            "box": box,
            "raw_category": raw_category,
            "image_category": image_category,
        })

    return detections

In [61]:
def build_categorical_row(categorical_columns, defaults, overrides):
    row = []
    for col in categorical_columns:
        if col in overrides:
            row.append(overrides[col])
        else:
            row.append(defaults.get(col, "Unknown"))
    return row


def encode_categorical(encoder, categorical_columns, row_values):
    df_row = pd.DataFrame([row_values], columns=categorical_columns)
    encoded = encoder.transform(df_row)
    encoded = np.asarray(encoded.todense()) if hasattr(encoded, "todense") else np.asarray(encoded)

    if encoded.shape[1] >= TARGET_CATEGORICAL_FEATURES:
        return encoded[:, :TARGET_CATEGORICAL_FEATURES]

    padding = np.zeros((encoded.shape[0], TARGET_CATEGORICAL_FEATURES - encoded.shape[1]), dtype=np.float32)
    return np.hstack([encoded, padding])


def build_feature_vector(image_category, raw_category, body_type, review_text, nlp, artifacts):
    visual_vec = get_category_vector(image_category, artifacts["category_features"])

    if review_text and nlp is not None:
        text_vec = artifacts["vectorizer"].transform([clean_review_text(review_text, nlp)]).toarray()
    else:
        text_vec = artifacts["vectorizer"].transform([""]).toarray()

    numeric_vec = artifacts["scaler"].transform([[
        artifacts["fallbacks"]["numeric"]["height_cm_median"],
        artifacts["fallbacks"]["numeric"]["weight_kg_median"],
        artifacts["fallbacks"]["numeric"]["age_median"],
    ]])

    overrides = {"body type": body_type, "category": raw_category}
    row_values = build_categorical_row(
        artifacts["categorical_columns"], artifacts["fallbacks"]["categorical"], overrides
    )
    categorical_vec = encode_categorical(artifacts["encoder"], artifacts["categorical_columns"], row_values)

    return np.hstack([visual_vec, text_vec, numeric_vec, categorical_vec])

In [62]:
def predict_one(image_category, raw_category, body_type, review_text, nlp, artifacts):
    """Shared prediction logic for both the 'garment detected' and
    'no garment detected, use fallback category' paths."""
    X_input = build_feature_vector(image_category, raw_category, body_type, review_text, nlp, artifacts)

    pred_encoded = artifacts["model"].predict(X_input)
    pred_label = artifacts["label_encoder"].inverse_transform(pred_encoded)[0]
    pred_proba = artifacts["model"].predict_proba(X_input)[0]

    return {
        "category": image_category,
        "predicted_fit": pred_label,
        "probabilities": dict(zip(artifacts["label_encoder"].classes_, pred_proba)),
    }


def run_model_one_pipeline(image_path, review_text=None,
                            fallback_image_category="Dresses", fallback_raw_category="dress"):
    check_required_files()

    # Load everything ONCE
    artifacts = load_inference_artifacts()
    processor, detection_model = build_yolos()
    pose_landmarker = build_pose_landmarker()
    nlp = build_spacy_nlp() if review_text else None

    body_type = estimate_body_type_from_photo(image_path, pose_landmarker)
    print(f"\nBody Type Detected: {body_type}")

    garments = detect_and_classify_all_garments(image_path, processor, detection_model, artifacts["category_features"])

    results = []
    if garments:
        print("\nGarments Detected:")
        for g in garments:
            prediction = predict_one(g["image_category"], g["raw_category"], body_type, review_text, nlp, artifacts)
            prediction["garment"] = g["fashionpedia_label"]
            prediction["confidence"] = g["confidence"]
            results.append(prediction)
            print(f"  - {prediction['category']} (YOLOS label: {prediction['garment']}) | "
                  f"Confidence: {prediction['confidence']*100:.1f}% | Predicted Fit: {prediction['predicted_fit']}")
    else:
        print("\nNo garments detected by YOLOS. Using fallback category.")
        prediction = predict_one(fallback_image_category, fallback_raw_category, body_type, review_text, nlp, artifacts)
        results.append(prediction)
        print(f"  - {prediction['category']} (fallback) | Predicted Fit: {prediction['predicted_fit']}")

    return {"body_type": body_type, "results": results}

In [71]:
if __name__ == "__main__":
    output = run_model_one_pipeline(r"..\data\test.jpg")
    print(output)

All required artifact files found in: ..\data


Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]


Body Type Detected: athletic

Garments Detected:
  - Tops (YOLOS label: top, t-shirt, sweatshirt) | Confidence: 63.7% | Predicted Fit: large
  - Dresses (YOLOS label: dress) | Confidence: 88.7% | Predicted Fit: large


C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


{'body_type': 'athletic', 'results': [{'category': 'Tops', 'predicted_fit': 'large', 'probabilities': {'fit': np.float32(0.0326391), 'large': np.float32(0.7413844), 'small': np.float32(0.22597647)}, 'garment': 'top, t-shirt, sweatshirt', 'confidence': 0.637}, {'category': 'Dresses', 'predicted_fit': 'large', 'probabilities': {'fit': np.float32(0.038040426), 'large': np.float32(0.72079456), 'small': np.float32(0.24116498)}, 'garment': 'dress', 'confidence': 0.887}]}


---
# Part 7 — Model Evaluation



In [72]:
# --- Added: compute the metrics the original print block assumed already existed ---
y_pred = best_xgb_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
f1 = f1_score(y_test, y_pred, average="weighted")

cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=label_encoder.classes_)

In [75]:
print("\n========================================")
print("       FINAL MODEL EVALUATION")
print("========================================")

print(f"Model: XGBoost")

print(f"\nBest Parameters:")
print(random_search.best_params_)

print(f"\nBest Cross Validation Score:")
print(f"{random_search.best_score_:.4f}")

print(f"\nTest Accuracy:")
print(f"{accuracy:.4f}")

print(f"\nPrecision:")
print(f"{precision:.4f}")

print(f"\nRecall:")
print(f"{recall:.4f}")

print(f"\nF1 Score:")
print(f"{f1:.4f}")

print(f"\nConfusion Matrix:")
print(cm)

print(f"\nFull Classification Report:")
print(report)


       FINAL MODEL EVALUATION
Model: XGBoost

Best Parameters:
{'subsample': 0.8, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}

Best Cross Validation Score:
0.7569

Test Accuracy:
0.7872

Precision:
0.7713

Recall:
0.7872

F1 Score:
0.7573

Confusion Matrix:
[[38096   769   811]
 [ 4686  2539   313]
 [ 4693   420  2614]]

Full Classification Report:
              precision    recall  f1-score   support

         fit       0.80      0.96      0.87     39676
       large       0.68      0.34      0.45      7538
       small       0.70      0.34      0.46      7727

    accuracy                           0.79     54941
   macro avg       0.73      0.55      0.59     54941
weighted avg       0.77      0.79      0.76     54941

